# GitHub Repository Metrics Dataset (5K+ Repos) — Complete EDA
> **5,500 repos with 29 features: stars, forks, CI/CD, community health** | [Dataset](https://www.kaggle.com/datasets/lorenzoscaturchio/github-repo-metrics)

**5,000 rows · 29 columns · `github_repos.csv`**

## Table of Contents
1. [Objective & Evaluation Plan](#objective)
2. [Setup & Data Loading](#setup)
3. [Data Overview](#overview)
4. [Missing Data Analysis](#missing)
5. [Numeric Distributions](#distributions)
6. [Categorical Analysis](#categorical)
7. [Correlation Analysis](#correlations)
8. [Target Analysis](#target)
9. [Evaluation Readiness](#evaluation)
10. [Key Findings](#findings)

## 1. Objective & Evaluation Plan <a id='objective'></a>

**Objective:** turn `GitHub Repository Metrics Dataset (5K+ Repos)` into a reproducible `unsupervised exploration + candidate regression/classification framing` workflow.

**Evaluation / validation:** use **define metric after target selection** with **hold out a small validation slice once a target is chosen**.

**Candidate target:** `not yet fixed`.

**Working hypothesis:** There is no obvious target yet, so the immediate goal is to surface hypotheses and shortlist modeling targets.

This section makes the modeling goal explicit so later findings can be interpreted in terms of metrics, trade-offs, and deployment limitations rather than isolated charts.

In [ ]:
TARGET_COL = 'not yet fixed'
MODELING_TASK = 'unsupervised exploration + candidate regression/classification framing'
PRIMARY_METRIC = 'define metric after target selection'
VALIDATION_PLAN = 'hold out a small validation slice once a target is chosen'

print("Objective framing")
print("-" * 60)
print(f"Rows available      : 5,000")
print(f"Candidate target    : {TARGET_COL}")
print(f"Modeling task       : {MODELING_TASK}")
print(f"Primary metric      : {PRIMARY_METRIC}")
print(f"Validation approach : {VALIDATION_PLAN}")

## 2. Setup & Data Loading <a id='setup'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

import os
DATA_DIR = '/kaggle/input/github-repo-metrics'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '.'

df = pd.read_csv(f'{DATA_DIR}/github_repos.csv')
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

## 3. Data Overview <a id='overview'></a>

In [ ]:
print("Column Types:")
print(df.dtypes.value_counts().to_string())
print(f"\nDuplicate rows: {df.duplicated().sum():,}")
print(f"Total missing values: {df.isnull().sum().sum():,}")
print()
df.describe().round(2)

## 4. Missing Data Analysis <a id='missing'></a>

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('percent', ascending=False)

if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.4)))
    colors = ['#e74c3c' if p > 20 else '#f39c12' if p > 5 else '#2ecc71'
              for p in missing_df['percent']]
    bars = ax.barh(missing_df.index, missing_df['percent'], color=colors, alpha=0.85)
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Data by Column', fontweight='bold')
    for bar, pct in zip(bars, missing_df['percent']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found — dataset is complete!")

## 5. Numeric Distributions <a id='distributions'></a>

In [ ]:
NUMERIC_COLS = ['stars', 'forks', 'watchers', 'open_issues', 'closed_issues', 'open_pull_requests', 'merged_pull_requests', 'contributors', 'commits', 'releases', 'readme_length', 'has_ci']

n = len(NUMERIC_COLS)
ncols = min(3, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
if n == 1:
    axes = [axes]
else:
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, col in enumerate(NUMERIC_COLS):
    ax = axes[i]
    ax.hist(df[col].dropna(), bins=40, color='#3498db', alpha=0.8, edgecolor='white', linewidth=0.3)
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_val:.2f}')
    ax.set_title(col, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for outlier detection
fig, axes = plt.subplots(1, min(len(NUMERIC_COLS), 6), figsize=(min(len(NUMERIC_COLS), 6) * 3, 5))
if min(len(NUMERIC_COLS), 6) == 1:
    axes = [axes]

for ax, col in zip(axes if hasattr(axes, '__iter__') else [axes], NUMERIC_COLS[:6]):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.6))
    ax.set_title(col, fontweight='bold', fontsize=10)

plt.suptitle('Box Plots — Outlier Detection', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Categorical Analysis <a id='categorical'></a>

In [ ]:
CAT_COLS = ['language', 'license', 'default_branch']

n = len(CAT_COLS)
ncols = min(2, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows))
if n == 1:
    axes = [axes]
else:
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

palette = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6',
           '#1abc9c', '#e67e22', '#34495e', '#16a085', '#c0392b']

for i, col in enumerate(CAT_COLS):
    ax = axes[i]
    counts = df[col].value_counts().head(10)
    bars = ax.barh(counts.index.astype(str), counts.values,
                   color=palette[:len(counts)], alpha=0.85)
    ax.set_title(f'{col} (top {min(10, len(counts))})', fontweight='bold')
    ax.set_xlabel('Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + max(counts) * 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Correlation Analysis <a id='correlations'></a>

In [ ]:
CORR_COLS = ['stars', 'forks', 'watchers', 'open_issues', 'closed_issues', 'open_pull_requests', 'merged_pull_requests', 'contributors', 'commits', 'releases', 'readme_length', 'has_ci', 'test_coverage', 'has_code_of_conduct', 'has_contributing_guide']

corr = df[CORR_COLS].corr()

fig, ax = plt.subplots(figsize=(min(12, len(CORR_COLS) + 2), min(10, len(CORR_COLS) + 1)))
mask = np.triu(np.ones_like(corr), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, mask=mask,
            annot_kws={'size': 9}, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Top absolute correlations (excluding self-correlations)
corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        corr_pairs.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print("Top 10 Feature Correlations:")
print(f"{'Feature A':<25} {'Feature B':<25} {'Correlation':>12}")
print("-" * 65)
for a, b, r in corr_pairs[:10]:
    arrow = '+' if r > 0 else '-'
    print(f"{a:<25} {b:<25} {arrow}{abs(r):>11.3f}")

## 8. Target Analysis <a id='target'></a>

*No standard target column detected. Explore potential targets manually.*

## 9. Evaluation Readiness <a id='evaluation'></a>

Before modeling, translate the EDA into a validation plan. This keeps metric choices, leakage checks, and leaderboard expectations aligned with the problem structure.

In [ ]:
TARGET_COL = 'manual selection required'
MODELING_TASK = 'unsupervised exploration + candidate regression/classification framing'
PRIMARY_METRIC = 'define metric after target selection'
VALIDATION_PLAN = 'hold out a small validation slice once a target is chosen'
HIGH_CARD_TEXT = 'description, topics, created_date, last_commit_date'

print("Evaluation readiness checklist")
print("-" * 60)
print(f"Target candidate         : {TARGET_COL}")
print(f"Modeling task            : {MODELING_TASK}")
print(f"Primary metric           : {PRIMARY_METRIC}")
print(f"Validation plan          : {VALIDATION_PLAN}")
print(f"High-cardinality columns : {HIGH_CARD_TEXT}")
print("\nRecommended baseline stack:")
if MODELING_TASK == "classification":
    print("- LogisticRegression / LightGBMClassifier with stratified validation")
elif MODELING_TASK == "regression":
    print("- Ridge / LightGBMRegressor with error analysis on residual tails")
else:
    print("- Clustering, retrieval, or weak-label experiments before supervised modeling")
print("\nRisk checks:")
print("- Verify no leakage columns encode the target directly")
print("- Compare train/validation distributions before reading leaderboard movement")
print("- Document trade-offs, caveats, and limitations before feature expansion")

## 10. Key Findings <a id='findings'></a>

In [ ]:
# Data quality summary
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)
print(f"  Rows:            {len(df):,}")
print(f"  Columns:         {df.shape[1]}")
print(f"  Duplicates:      {df.duplicated().sum():,}")
print(f"  Total missing:   {df.isnull().sum().sum():,}")
print(f"  Complete rows:   {df.dropna().shape[0]:,} ({df.dropna().shape[0]/len(df)*100:.1f}%)")
print()

numeric_cols = df.select_dtypes(include=['number']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns
print(f"  Numeric columns:     {len(numeric_cols)}")
print(f"  Categorical columns: {len(cat_cols)}")
print()

if len(numeric_cols) > 0:
    print("Potential outliers (values > 3 std from mean):")
    for col in numeric_cols[:10]:
        z = (df[col] - df[col].mean()) / df[col].std()
        outliers = (z.abs() > 3).sum()
        if outliers > 0:
            print(f"  {col}: {outliers:,} rows ({outliers/len(df)*100:.1f}%)")

### Insights, Trade-offs, and Next Steps
- **Insight:** the strongest opportunity usually comes from the small set of columns with the clearest business interpretation.
- **Observation:** missingness, skew, and high-cardinality text fields often drive the most useful feature engineering decisions.
- **Trade-off:** richer feature sets can improve metrics, but they also increase leakage risk and maintenance cost.
- **Limitation:** EDA alone cannot prove causality, so validation and error analysis should confirm every major hypothesis.
- **Hypothesis:** targeted feature engineering on the most informative fields should outperform a naive all-columns baseline.

### Next Steps
- Feature engineering: create interaction terms from correlated features
- Handle missing values: imputation strategy depends on missingness pattern (MCAR/MAR/MNAR)
- Scale numeric features before modeling (StandardScaler or RobustScaler for outliers)
- Encode categoricals: one-hot for low cardinality, target encoding for high cardinality
- Try baseline models: LogisticRegression/Ridge for linear, XGBoost/LightGBM for tree-based